DATA PRE PROCESSING

In [13]:
#!pip install matplotlib
#!pip install seaborn
#!pip install statsmodels
#!pip install ipywidgets

In [95]:
#Loading Data
import pandas as pd

df = pd.read_csv(
    "motor_input.csv",
    sep=";",
    quotechar='"',
    skipinitialspace=True
)


In [ ]:
#Checking DataType of columns - Verified to be Correct
df.info()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Assume df is already loaded

# Numeric columns (exclude ID if you like)
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
if "insured_id" in num_cols:
    num_cols.remove("insured_id")

# One histogram per numeric column
for col in num_cols:
    plt.figure(figsize=(6, 4))
    data = df[col].dropna()
    if data.nunique() > 1:
        plt.hist(data, bins=30, edgecolor="black")
    else:
        # If almost all values are the same, use very few bins
        plt.hist(data, bins=10, edgecolor="black")
    plt.title(f"Histogram of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
if "insured_id" in num_cols:
    num_cols.remove("insured_id")

for col in num_cols:
    plt.figure(figsize=(6, 4))
    data = df[col].dropna()
    sns.histplot(data, kde=True, bins=30, edgecolor="black")
    plt.title(f"Histogram + KDE: {col}")
    plt.xlabel(col)
    plt.ylabel("Density")
    plt.tight_layout()
    plt.show()

In [ ]:
# Categorical columns
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
if "insured_id" in cat_cols:
    cat_cols.remove("insured_id")

# One frequency bar chart per categorical column
for col in cat_cols:
    plt.figure(figsize=(6, 4))
    counts = df[col].value_counts().sort_index()
    counts.plot(kind="bar", edgecolor="black")
    plt.title(f"Frequency of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [96]:
# Missing Values Analysis

# 1. Count missing values per column
missing_counts = df.isna().sum()
print("Missing values per column:")
print(missing_counts)
print()

# 2. Total rows before dropping
print("Rows before:", len(df))

# 3. Rows that are fully complete (no missing values in any column)
rows_fully_complete = df.dropna().shape[0]
print("Rows fully complete (no NaNs):", rows_fully_complete)
print()

# 4. Drop rows with any missing values
df_clean = df.dropna().copy()

# 5. Reset index (optional but useful)
df_clean = df_clean.reset_index(drop=True)

# 6. Replace original df with cleaned version
df = df_clean

print("Rows after dropping any missing:", len(df))

print("################################")
print(df.isna().sum())

Missing values per column:
insured_id                        0
year                              0
policy_type                       0
policy_status                     0
business_type                     0
payment_frequency                 0
bonus_score                       0
driver_age                        0
vehicle_age                       2
age_driving_licence               2
fuel_type                      1287
vehicle_value                   513
seats                             0
power_to_weight_ratio             0
vehicle_brand                     0
municipality_type                 0
circulation_area                  0
total_premium                     0
liability_premium                 0
property_damage_premium           0
theft_premium                     0
fire_premium                      0
glass_premium                     0
legal_protection_premium          0
occupants_premium                 0
total_claims                      0
liability_claims                  0
l

In [97]:
#Outlier and extreme‑value diagnostics 

import numpy as np
import pandas as pd

num_vars = [
    "driver_age", "vehicle_age", "vehicle_value",
    "power_to_weight_ratio", "total_exposure",
    "total_claims", "total_incurred"
]

# Basic stats
desc = df[num_vars].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])

# Compute IQR for each variable
Q1 = df[num_vars].quantile(0.25)
Q3 = df[num_vars].quantile(0.75)
IQR = Q3 - Q1

# Append IQR as a new row
desc.loc["IQR"] = IQR

# Count outliers above upper IQR fence (Q3 + 1.5*IQR)
upper_fence = Q3 + 1.5 * IQR
outlier_counts = (df[num_vars] > upper_fence).sum()

# Append outlier count as a new row
desc.loc["outliers_above_IQR"] = outlier_counts

desc

,driver_age,vehicle_age,vehicle_value,power_to_weight_ratio,total_exposure,total_claims,total_incurred
count,352338.000,352338.000,352338.000,352338.000,352338.000,352338.000,352338.000
mean,48.990,25.905,26937.255,12.337,0.707,0.217,208.353
std,12.068,11.614,12282.259,2.470,0.326,0.658,2279.210
min,18.000,0.000,1630.125,0.000,0.000,0.000,0.000
1%,27.000,4.000,10152.853,7.040,0.019,0.000,0.000
5%,31.000,8.000,12751.200,8.760,0.090,0.000,0.000
50%,48.000,25.000,24687.338,12.060,0.838,0.000,0.000
95%,69.000,46.000,49322.221,16.710,1.000,1.000,1050.479
99%,75.000,51.000,71121.750,18.930,1.000,3.000,3780.948
max,90.000,68.000,374034.210,64.810,1.000,18.000,571128.318


In [98]:
#Sanity checks and basic validity filters

# 1. Positive exposure
df = df[df["total_exposure"] > 0].copy()

# 2. Non-negative claim counts (example for total_claims; extend as needed)
df = df[(df["total_claims"] >= 0) & (df["total_claims"] <= 5)].copy()

# 3. Non-negative incurred amounts
df = df[(df["total_incurred"] >= 0) & (df["total_incurred"] <= 4000)].copy()

# 4. Reasonable driver_age and vehicle_age
df = df[(df["driver_age"] >= 18) & (df["driver_age"] <= 100)].copy()
df = df[(df["vehicle_age"] >= 0) & (df["vehicle_age"] <= 70)].copy()

# 5. Reasonable Vehicle Value
df = df[(df["vehicle_value"] >= 0) & (df["vehicle_value"] <= 80000)].copy()

# 6. Reasonable Power to Weight Ratio
df = df[(df["power_to_weight_ratio"] >= 0) & (df["power_to_weight_ratio"] <= 20)].copy()




# Optional: reset index again after filtering
df = df.reset_index(drop=True)

print("Rows after filtering:", len(df))

Rows after filtering: 345083


BUILDING GLM

In [99]:
# 1) Count cars per brand
brand_counts = (
    df.groupby("vehicle_brand")
    .size()
    .sort_values(ascending=False)
)

print(brand_counts.head(15))  # inspect top 15

# 2) Pick top 10
k = 10
top_brands = list(brand_counts.head(k).index)

# 3) Create grouped brand variable
def group_brand(brand):
    return brand if brand in top_brands else "OTHER"

df["vehicle_brand_grp"] = df["vehicle_brand"].apply(group_brand)

vehicle_brand
RENAULT       33857
CITROEN       32098
PEUGEOT       29202
VOLKSWAGEN    27974
FORD          27780
SEAT          27650
OPEL          27553
AUDI          15884
MERCEDES      14287
TOYOTA        13953
BMW           13279
NISSAN        12987
HYUNDAI        9156
FIAT           6780
KIA            6069
dtype: int64


In [100]:
predictor_cols = [
    "driver_age",
    "vehicle_age",
    "age_driving_licence",
    "fuel_type",
    "vehicle_value",
    "seats",
    "power_to_weight_ratio",
    "vehicle_brand_grp",
    "municipality_type",
    "circulation_area"
]

premium_targets = [
    "total_premium",
    "liability_premium",
    "property_damage_premium",
    "theft_premium",
    "fire_premium",
    "glass_premium",
    "legal_protection_premium",
    "occupants_premium"
]

frequency_targets = [
    "total_claims",
    "liability_claims",
    "property_claims",
    "theft_claims",
    "fire_claims",
    "glass_claims",
    "legal_protection_claims",
    "occupants_claims"
]

severity_targets = [
    "total_incurred",
    "liability_incurred",
    "property_incurred",
    "theft_incurred",
    "fire_incurred",
    "glass_incurred",
    "legal_protection_incurred",
    "occupants_incurred"
]

# Exposure mapping for each target
exposure_map = {
    # Frequency
    "total_claims": "total_exposure",
    "liability_claims": "liability_exposure",
    "property_claims": "total_exposure",
    "theft_claims": "total_exposure",
    "fire_claims": "total_exposure",
    "glass_claims": "total_exposure",
    "legal_protection_claims": "total_exposure",
    "occupants_claims": "total_exposure",

    # Pure premium (your "severity")
    "total_incurred": "total_exposure",
    "liability_incurred": "liability_exposure",
    "property_incurred": "total_exposure",
    "theft_incurred": "total_exposure",
    "fire_incurred": "total_exposure",
    "glass_incurred": "total_exposure",
    "legal_protection_incurred": "total_exposure",
    "occupants_incurred": "total_exposure",

    # Premium models
    "total_premium": "total_exposure",
    "liability_premium": "liability_exposure",
    "property_damage_premium": "total_exposure",
    "theft_premium": "total_exposure",
    "fire_premium": "total_exposure",
    "glass_premium": "total_exposure",
    "legal_protection_premium": "total_exposure",
    "occupants_premium": "total_exposure",
}

In [101]:
#FREQUENCY MODELING DISRIBUTION DECISION
import pandas as pd
import numpy as np

frequency_targets = [
    "total_claims",
    "liability_claims",
    "property_claims",
    "theft_claims",
    "fire_claims",
    "glass_claims",
    "legal_protection_claims",
    "occupants_claims"
]

# Optional: threshold for overdispersion
# overdispersion = var / mean
# <= thresh  → Poisson
# >  thresh  → Negative Binomial
OVERDISPERSION_THRESHOLD = 1.5

results = []

for col in frequency_targets:
    y = df[col]
    
    mean_counts = y.mean()
    var_counts = y.var()  # pandas uses ddof=1 by default (sample variance)
    
    # Overdispersion ratio: for Poisson, var ≈ mean, so ratio ≈ 1
    overdispersion = var_counts / mean_counts if mean_counts > 0 else np.nan
    
    if np.isnan(overdispersion) or mean_counts == 0:
        suggestion = "uncertain (very low/zero mean)"
    elif overdispersion <= OVERDISPERSION_THRESHOLD:
        suggestion = "Poisson"
    else:
        suggestion = "Negative Binomial"
    
    results.append({
        "target": col,
        "mean": mean_counts,
        "variance": var_counts,
        "overdispersion": overdispersion,
        "suggested_distribution": suggestion
    })

summary_df = pd.DataFrame(results)
summary_df

,target,mean,variance,overdispersion,suggested_distribution
0,total_claims,0.187,0.316,1.688,Negative Binomial
1,liability_claims,0.091,0.110,1.209,Poisson
2,property_claims,0.051,0.123,2.414,Negative Binomial
3,theft_claims,0.005,0.005,1.030,Poisson
4,fire_claims,0.000,0.000,1.000,Poisson
5,glass_claims,0.032,0.033,1.043,Poisson
6,legal_protection_claims,0.008,0.008,1.013,Poisson
7,occupants_claims,0.000,0.000,1.000,Poisson


In [102]:
#SEVERITY MODELING DISRIBUTION DECISION

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st
import statsmodels.api as sm
import seaborn as sns

severity_targets = [
    "total_incurred",
    "liability_incurred",
    "property_incurred",
    "theft_incurred",
    "fire_incurred",
    "glass_incurred",
    "legal_protection_incurred",
    "occupants_incurred"
]

sns.set(style="whitegrid", font_scale=1.0)

results = []

for col in severity_targets:
    y = df[col]
    y = y[y > 0]
    
    if len(y) < 10:
        results.append({
            "target": col,
            "n_obs": len(y),
            "ll_gamma": np.nan,
            "ll_lognorm": np.nan,
            "ll_diff": np.nan,
            "preferred_distribution": "too few data"
        })
        continue
    
    # # ---------- 1. Simple plots ----------
    # fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # # Severity histogram
    # axes[0].hist(y, bins=50, edgecolor="black", alpha=0.7)
    # axes[0].set_title(f"{col}: severity")
    # axes[0].set_xlabel("severity")
    # axes[0].set_ylabel("count")
    
    # # Log-severity histogram
    log_y = np.log(y)
    # axes[1].hist(log_y, bins=50, edgecolor="black", alpha=0.7)
    # axes[1].set_title(f"{col}: log(severity)")
    # axes[1].set_xlabel("log(severity)")
    # axes[1].set_ylabel("count")
    
    # # Q-Q plot of log(severity) vs Normal
    # sm.qqplot(log_y, line="45", ax=axes[2], fit=True)
    # axes[2].set_title(f"{col}: Q-Q of log(severity)")
    
    # plt.tight_layout()
    # plt.show()
    
    # ---------- 2. Fit univariate Gamma and Log-Normal ----------
    # Gamma MLE
    gamma_params = st.gamma.fit(y)  # (a, loc, scale)
    ll_gamma = np.sum(st.gamma.logpdf(y, *gamma_params))
    
    # Log-Normal: fit Normal to log(y)
    lognorm_params = st.norm.fit(log_y)  # (mu, sigma)
    mu, sigma = lognorm_params
    ll_lognorm = np.sum(st.norm.logpdf(log_y, mu, sigma) - np.log(y))
    
    ll_diff = ll_lognorm - ll_gamma
    
    if ll_diff > 0:
        pref = "Log-Normal"
    elif ll_diff < 0:
        pref = "Gamma"
    else:
        pref = "Either (similar LL)"
    
    results.append({
        "target": col,
        "n_obs": len(y),
        "ll_gamma": ll_gamma,
        "ll_lognorm": ll_lognorm,
        "ll_diff": ll_diff,
        "preferred_distribution": pref
    })

severity_summary = pd.DataFrame(results)

# ---------- 3. Clean table ----------
# Show only the clean decision table, no scientific notation
pd.set_option("display.float_format", "{:.3f}".format)
clean_table = severity_summary[
    ["target", "n_obs", "ll_gamma", "ll_lognorm", "ll_diff", "preferred_distribution"]
].copy()

clean_table

,target,n_obs,ll_gamma,ll_lognorm,ll_diff,preferred_distribution
0,total_incurred,37908,-297552.738,-302050.516,-4497.778,Gamma
1,liability_incurred,17480,-136927.305,-142640.435,-5713.130,Gamma
2,property_incurred,10674,-84942.917,-87295.037,-2352.121,Gamma
3,theft_incurred,1685,-188954.950,-11979.615,176975.335,Log-Normal
4,fire_incurred,68,-537.246,-555.662,-18.416,Gamma
5,glass_incurred,10519,-1563767.763,-72452.570,1491315.192,Log-Normal
6,legal_protection_incurred,2788,-18361.537,-18956.507,-594.971,Gamma
7,occupants_incurred,13,-763.063,-91.131,671.933,Log-Normal


In [103]:
import pandas as pd
import numpy as np

# 1. Define predictor columns
predictor_cols = [
    "driver_age",
    "vehicle_age",
    "age_driving_licence",
    "fuel_type",
    "vehicle_value",
    "seats",
    "power_to_weight_ratio",
    "vehicle_brand_grp",
    "municipality_type",
    "circulation_area"
]

# 2. Split into numeric and categorical
cat_cols = ["fuel_type", "vehicle_brand_grp", "municipality_type", "circulation_area"]
num_cols = [c for c in predictor_cols if c not in cat_cols]

# 3. Ensure numeric predictors are numeric
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# 4. Ensure categorical predictors are categorical (for statsmodels formula)
for col in cat_cols:
    df[col] = df[col].astype("string").astype("category")

# 5. Final predictor DataFrame
predictor_df = df[predictor_cols].copy()

# Optional: quick check
print(predictor_df.dtypes)
print(predictor_df.head())

driver_age                  int64
vehicle_age               float64
age_driving_licence       float64
fuel_type                category
vehicle_value             float64
seats                       int64
power_to_weight_ratio     float64
vehicle_brand_grp        category
municipality_type        category
circulation_area         category
dtype: object
   driver_age  vehicle_age  age_driving_licence fuel_type  vehicle_value  \
0          43       24.000               19.000         D      65065.123   
1          44       25.000               19.000         D      65065.123   
2          45       26.000               18.000         D      65065.123   
3          45       26.000               19.000         D      51626.662   
4          46       27.000               19.000         D      51626.662   

   seats  power_to_weight_ratio vehicle_brand_grp municipality_type  \
0      8                 10.150          MERCEDES                 I   
1      8                 10.150          MERCED

FREQUENCY GLMS

In [104]:

import numpy as np

# Create log-exposure columns for each coverage
df["log_total_exposure"] = np.log(np.clip(df["total_exposure"], a_min=1e-12, a_max=None))
df["log_liability_exposure"] = np.log(np.clip(df["liability_exposure"], a_min=1e-12, a_max=None))
# add others if you have separate exposures for other coverages

In [105]:

import numpy as np
import pickle
import os
import statsmodels.formula.api as smf
from statsmodels.genmod.families import Poisson, NegativeBinomial

# Random train–test split (run once)
test_frac   = 0.2    # 80/20 split
random_seed = 42

rng = np.random.default_rng(random_seed)
mask = rng.random(len(df)) < (1 - test_frac)

df_train = df[mask].copy()
df_test  = df[~mask].copy()

# IMPORTANT: make sure predictor_cols does NOT include log-exposure columns
# predictor_cols = [c for c in predictor_cols if c not in ["log_total_exposure", "log_liability_exposure"]]

# In-memory containers
glm_models = {}
glm_diagnostics = {}

# Directory for model files
model_dir = r"D:\2_Research\FILES\2 Learning\GLM\glm_model_files"
os.makedirs(model_dir, exist_ok=True)

# Frequency configuration: response -> (exposure, family)
freq_config = {
    "total_claims":           ("total_exposure",        NegativeBinomial()),
    "liability_claims":       ("liability_exposure",    Poisson()),
    "property_claims":        ("total_exposure",        NegativeBinomial()),
    "theft_claims":           ("total_exposure",        Poisson()),
    "fire_claims":            ("total_exposure",        Poisson()),
    "glass_claims":           ("total_exposure",        Poisson()),
    "legal_protection_claims":("total_exposure",        Poisson()),
    "occupants_claims":       ("total_exposure",        Poisson()),
}

# Mapping from exposure to offset column (already created in your data)
offset_map = {
    "total_exposure":     "log_total_exposure",
    "liability_exposure": "log_liability_exposure",
    # add more if you have extra exposure columns with their log versions
}

c:\Users\eswar\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\genmod\families\family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
c:\Users\eswar\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\genmod\families\family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


In [106]:

def build_freq_glm(response_col: str,
                   exposure_col: str,
                   family,
                   df_train,
                   df_test,
                   predictor_cols,
                   glm_models,
                   glm_diagnostics,
                   model_dir,
                   offset_map):
    """
    Build a frequency GLM for one response, with:
    - train fit
    - diagnostics
    - test metrics
    - storage in memory and on disk
    """

    # Derive offset column from exposure
    offset_col = offset_map[exposure_col]

    # Unique key and label
    model_key   = f"freq_{response_col}"
    model_label = f"Frequency: {response_col}"

    # Build formula string (no offset in the formula; offset passed separately)
    freq_formula = f"{response_col} ~ " + " + ".join(predictor_cols)
    #freq_formula = f"{response_col} ~ driver_age + vehicle_age + age_driving_licence + fuel_type + vehicle_value + seats + power_to_weight_ratio + vehicle_brand_grp + municipality_type + circulation_area"

    # -------- FIT MODEL ON TRAIN --------
    freq_model = smf.glm(
        formula=freq_formula,
        data=df_train,
        family=family,
        offset=df_train[offset_col]
    ).fit()

    # -------- TRAIN-SIDE DIAGNOSTICS --------
    train_AIC        = freq_model.aic
    train_deviance   = freq_model.deviance
    train_df_resid   = freq_model.df_resid
    train_pearson    = freq_model.pearson_chi2
    train_dispersion = train_pearson / train_df_resid

    # -------- TEST-SIDE PREDICTIONS --------
    y_test    = df_test[response_col].values
    yhat_test = freq_model.predict(
        df_test,
        offset=df_test[offset_col]
    ).values

    # Safety check: counts should be non-negative for Poisson/NB
    if isinstance(family, (Poisson, NegativeBinomial)):
        if np.any(yhat_test < 0):
            raise ValueError(f"Negative predicted counts for {model_key}; check link/offset/predictors.")

    # MAE and RMSE on counts
    freq_mae_count  = np.mean(np.abs(y_test - yhat_test))
    freq_rmse_count = np.sqrt(np.mean((y_test - yhat_test) ** 2))

    # Frequency per unit exposure
    f_test    = y_test / df_test[exposure_col].values
    fhat_test = yhat_test / df_test[exposure_col].values

    freq_mae_freq  = np.mean(np.abs(f_test - fhat_test))
    freq_rmse_freq = np.sqrt(np.mean((f_test - fhat_test) ** 2))

    # -------- CLEAN DIAGNOSTICS PRINT --------
    print("=" * 80)
    print(f"Model key   : {model_key}")
    print(f"Model label : {model_label}")
    print(f"Family      : {family.__class__.__name__}")
    print("-" * 80)
    print("TRAIN DIAGNOSTICS")
    print(f"  AIC               : {train_AIC:,.3f}")
    print(f"  Deviance          : {train_deviance:,.3f}")
    print(f"  Pearson chi2/df   : {train_dispersion:,.3f}")
    print("-" * 80)
    print("TEST METRICS (Frequency)")
    print(f"  MAE_count         : {freq_mae_count:.6f}")
    print(f"  RMSE_count        : {freq_rmse_count:.6f}")
    print(f"  MAE_freq          : {freq_mae_freq:.6f}")
    print(f"  RMSE_freq         : {freq_rmse_freq:.6f}")
    print("=" * 80)

    # -------- STORE MODEL IN MEMORY --------
    glm_models[model_key] = freq_model

    # -------- STORE DIAGNOSTICS IN MEMORY --------
    glm_diagnostics[model_key] = {
        "model_label": model_label,
        "family": family.__class__.__name__,
        "type": "frequency",
        "response_col": response_col,
        "exposure_col": exposure_col,
        "offset_col": offset_col,
        # train-side
        "train_AIC": train_AIC,
        "train_deviance": train_deviance,
        "train_df_resid": train_df_resid,
        "train_pearson_chi2": train_pearson,
        "train_dispersion": train_dispersion,
        # test-side
        "test_MAE_count": freq_mae_count,
        "test_RMSE_count": freq_rmse_count,
        "test_MAE_freq": freq_mae_freq,
        "test_RMSE_freq": freq_rmse_freq,
    }

    # -------- STORE MODEL ON DISK --------
    filepath = os.path.join(model_dir, f"{model_key}.pkl")
    with open(filepath, "wb") as f:
        pickle.dump(freq_model, f)




In [107]:
# Fit all frequency models
for response_col, (exposure_col, family) in freq_config.items():
    build_freq_glm(
        response_col=response_col,
        exposure_col=exposure_col,
        family=family,
        df_train=df_train,
        df_test=df_test,
        predictor_cols=predictor_cols,
        glm_models=glm_models,
        glm_diagnostics=glm_diagnostics,
        model_dir=model_dir,
        offset_map=offset_map,
    )

Model key   : freq_total_claims
Model label : Frequency: total_claims
Family      : NegativeBinomial
--------------------------------------------------------------------------------
TRAIN DIAGNOSTICS
  AIC               : 272,326.322
  Deviance          : 156,350.417
  Pearson chi2/df   : 1.432
--------------------------------------------------------------------------------
TEST METRICS (Frequency)
  MAE_count         : 0.310501
  RMSE_count        : 0.549442
  MAE_freq          : 0.460350
  RMSE_freq         : 1.366353
Model key   : freq_liability_claims
Model label : Frequency: liability_claims
Family      : Poisson
--------------------------------------------------------------------------------
TRAIN DIAGNOSTICS
  AIC               : 170,024.593
  Deviance          : 124,356.208
  Pearson chi2/df   : 1.254
--------------------------------------------------------------------------------
TEST METRICS (Frequency)
  MAE_count         : 0.164564
  RMSE_count        : 0.328937
  MAE_freq 

SEVERITY GLMS

In [108]:
#1. Severity configuration

import numpy as np
import pickle
import os
import statsmodels.formula.api as smf
from statsmodels.genmod.families import Gamma, Gaussian

severity_targets = [
    "total_incurred",
    "liability_incurred",
    "property_incurred",
    "theft_incurred",
    "fire_incurred",
    "glass_incurred",
    "legal_protection_incurred",
    "occupants_incurred",
]

# Map incurred column to its corresponding claims column for filtering
severity_claims_map = {
    "total_incurred":               "total_claims",
    "liability_incurred":           "liability_claims",
    "property_incurred":            "property_claims",
    "theft_incurred":               "theft_claims",
    "fire_incurred":                "fire_claims",
    "glass_incurred":               "glass_claims",
    "legal_protection_incurred":    "legal_protection_claims",
    "occupants_incurred":           "occupants_claims",
}

# Preferred GLM family for each severity target
severity_family_map = {
    "total_incurred":               Gamma(),   # Log-Normal
    "liability_incurred":           Gamma(),   # Log-Normal
    "property_incurred":            Gamma(),   # Log-Normal
    "theft_incurred":               Gamma(),   # Log-Normal
    "fire_incurred":                Gamma(),   # Log-Normal
    "glass_incurred":               Gamma(),      # Gamma on raw severity
    "legal_protection_incurred":    Gamma(),   # Log-Normal
    "occupants_incurred":           Gamma(),      # Gamma on raw severity
}

In [109]:
def build_sev_glm(target_incurred: str,
                  df_train,
                  df_test,
                  predictor_cols,
                  glm_models,
                  glm_diagnostics,
                  model_dir,
                  severity_claims_map,
                  severity_family_map):
    """
    Build a severity GLM for one incurred target, with:
    - claims-only and incurred>0 filtering
    - automatic per-claim severity calculation
    - automatic log transform for Gaussian (Log-Normal) cases
    - safety checks for NaN/Inf/non-positive values
    - train fit, diagnostics, test MAE/RMSE
    - storage in memory and on disk
    """

    claims_col = severity_claims_map[target_incurred]
    family     = severity_family_map[target_incurred]

    model_key   = f"sev_{target_incurred}"
    model_label = f"Severity: {target_incurred}"

    # -------- TRAIN / TEST CLAIMS-ONLY AND INCURRED>0 SUBSETS --------
    mask_train = (
        (df_train[claims_col] > 0) &
        (df_train[target_incurred] > 0)
    )
    mask_test = (
        (df_test[claims_col] > 0) &
        (df_test[target_incurred] > 0)
    )

    df_train_sev = df_train[mask_train].copy()
    df_test_sev  = df_test[mask_test].copy()

    # -------- CONSTRUCT PER-CLAIM SEVERITY ON COPIES --------
    train_sev_pc = df_train_sev[target_incurred] / df_train_sev[claims_col]
    test_sev_pc  = df_test_sev[target_incurred] / df_test_sev[claims_col]

    if isinstance(family, Gaussian):
        # Log-Normal case: use log(severity per claim)
        # Ensure strictly positive before log
        mask_train_pos = train_sev_pc > 0
        mask_test_pos  = test_sev_pc > 0

        df_train_sev = df_train_sev[mask_train_pos].copy()
        df_test_sev  = df_test_sev[mask_test_pos].copy()

        train_sev_pc = train_sev_pc[mask_train_pos]
        test_sev_pc  = test_sev_pc[mask_test_pos]

        # Log transform
        df_train_sev["response_sev"] = np.log(train_sev_pc)
        df_test_sev["response_sev"]  = np.log(test_sev_pc)

        response_desc = "log(severity per claim)"
    else:
        # Gamma case: use raw per-claim severity
        # Ensure strictly positive
        mask_train_pos = train_sev_pc > 0
        mask_test_pos  = test_sev_pc > 0

        df_train_sev = df_train_sev[mask_train_pos].copy()
        df_test_sev  = df_test_sev[mask_test_pos].copy()

        train_sev_pc = train_sev_pc[mask_train_pos]
        test_sev_pc  = test_sev_pc[mask_test_pos]

        df_train_sev["response_sev"] = train_sev_pc
        df_test_sev["response_sev"]  = test_sev_pc

        response_desc = "severity per claim (raw)"

    response_col = "response_sev"

    # -------- OPTIONAL: DROP NON-FINITE VALUES (SAFETY) --------
    mask_train_finite = np.isfinite(df_train_sev[response_col])
    mask_test_finite  = np.isfinite(df_test_sev[response_col])

    df_train_sev = df_train_sev[mask_train_finite].copy()
    df_test_sev  = df_test_sev[mask_test_finite].copy()

    if len(df_train_sev) < 10:
        print(f"[WARNING] {target_incurred}: very few training rows after filtering ({len(df_train_sev)})")
        if len(df_train_sev) == 0:
            print(f"[SKIP] {target_incurred}: no training data after filtering; skipping model.")
            return

    # Build formula string
    sev_formula = f"{response_col} ~ " + " + ".join(predictor_cols)
    #sev_formula = f"{response_col} ~ driver_age + vehicle_age + age_driving_licence + fuel_type + vehicle_value + seats + power_to_weight_ratio + vehicle_brand_grp + municipality_type + circulation_area"

    # -------- FIT MODEL ON TRAIN CLAIMS --------
    sev_model = smf.glm(
        formula=sev_formula,
        data=df_train_sev,
        family=family
    ).fit()

    # -------- TRAIN-SIDE DIAGNOSTICS --------
    train_AIC       = sev_model.aic
    train_deviance  = sev_model.deviance
    train_df_resid  = sev_model.df_resid
    train_pearson   = sev_model.pearson_chi2
    train_dispersion = train_pearson / train_df_resid

    # -------- TEST-SIDE PREDICTIONS --------
    s_test    = df_test_sev[response_col].values
    shat_test = sev_model.predict(df_test_sev).values

    # MAE and RMSE on modeled scale
    sev_mae   = np.mean(np.abs(s_test - shat_test))
    sev_rmse  = np.sqrt(np.mean((s_test - shat_test) ** 2))

    # -------- CLEAN DIAGNOSTICS PRINT --------
    print("=" * 80)
    print(f"Model key   : {model_key}")
    print(f"Model label : {model_label}")
    print(f"Family      : {family.__class__.__name__}")
    print(f"Response    : {response_desc}")
    print(f"Train rows  : {len(df_train_sev)}")
    print(f"Test rows   : {len(df_test_sev)}")
    print("-" * 80)
    print("TRAIN DIAGNOSTICS")
    print(f"  AIC               : {train_AIC:,.3f}")
    print(f"  Deviance          : {train_deviance:,.3f}")
    print(f"  Pearson chi2/df   : {train_dispersion:,.3f}")
    print("-" * 80)
    print("TEST METRICS (Severity, modeled scale)")
    print(f"  MAE_sev           : {sev_mae:,.6f}")
    print(f"  RMSE_sev          : {sev_rmse:,.6f}")
    print("=" * 80)

    # -------- STORE MODEL IN MEMORY --------
    glm_models[model_key] = sev_model

    # -------- STORE DIAGNOSTICS IN MEMORY --------
    glm_diagnostics[model_key] = {
        "model_label": model_label,
        "family": family.__class__.__name__,
        "type": "severity",
        "incurred_col": target_incurred,
        "claims_col": claims_col,
        "response_desc": response_desc,
        "n_train": len(df_train_sev),
        "n_test": len(df_test_sev),
        # train-side
        "train_AIC": train_AIC,
        "train_deviance": train_deviance,
        "train_df_resid": train_df_resid,
        "train_pearson_chi2": train_pearson,
        "train_dispersion": train_dispersion,
        # test-side
        "test_MAE_sev": sev_mae,
        "test_RMSE_sev": sev_rmse,
    }

    # -------- STORE MODEL ON DISK --------
    filepath = os.path.join(model_dir, f"{model_key}.pkl")
    with open(filepath, "wb") as f:
        pickle.dump(sev_model, f)

In [110]:
#3. Loop through all severity GLMs

for target_incurred in severity_targets:
    build_sev_glm(
        target_incurred=target_incurred,
        df_train=df_train,
        df_test=df_test,
        predictor_cols=predictor_cols,
        glm_models=glm_models,
        glm_diagnostics=glm_diagnostics,
        model_dir=model_dir,
        severity_claims_map=severity_claims_map,
        severity_family_map=severity_family_map,
    )

c:\Users\eswar\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:308: DomainWarning: The InversePower link function does not respect the domain of the Gamma family.
  warnings.warn((f"The {type(family.link).__name__} link function "


Model key   : sev_total_incurred
Model label : Severity: total_incurred
Family      : Gamma
Response    : severity per claim (raw)
Train rows  : 30364
Test rows   : 7544
--------------------------------------------------------------------------------
TRAIN DIAGNOSTICS
  AIC               : 454,748.371
  Deviance          : 22,024.626
  Pearson chi2/df   : 0.572
--------------------------------------------------------------------------------
TEST METRICS (Severity, modeled scale)
  MAE_sev           : 392.165118
  RMSE_sev          : 516.268125


c:\Users\eswar\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:308: DomainWarning: The InversePower link function does not respect the domain of the Gamma family.
  warnings.warn((f"The {type(family.link).__name__} link function "
c:\Users\eswar\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:308: DomainWarning: The InversePower link function does not respect the domain of the Gamma family.
  warnings.warn((f"The {type(family.link).__name__} link function "


Model key   : sev_liability_incurred
Model label : Severity: liability_incurred
Family      : Gamma
Response    : severity per claim (raw)
Train rows  : 13982
Test rows   : 3498
--------------------------------------------------------------------------------
TRAIN DIAGNOSTICS
  AIC               : 217,556.208
  Deviance          : 7,105.009
  Pearson chi2/df   : 0.285
--------------------------------------------------------------------------------
TEST METRICS (Severity, modeled scale)
  MAE_sev           : 338.291640
  RMSE_sev          : 501.904723
Model key   : sev_property_incurred
Model label : Severity: property_incurred
Family      : Gamma
Response    : severity per claim (raw)
Train rows  : 8536
Test rows   : 2138
--------------------------------------------------------------------------------
TRAIN DIAGNOSTICS
  AIC               : 129,303.118
  Deviance          : 5,646.465
  Pearson chi2/df   : 0.664
---------------------------------------------------------------------------

c:\Users\eswar\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:308: DomainWarning: The InversePower link function does not respect the domain of the Gamma family.
  warnings.warn((f"The {type(family.link).__name__} link function "
c:\Users\eswar\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:308: DomainWarning: The InversePower link function does not respect the domain of the Gamma family.
  warnings.warn((f"The {type(family.link).__name__} link function "
c:\Users\eswar\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:308: DomainWarning: The InversePower link function does not respect the domain of the Gamma family.
  warnings.warn((f"The {type(family.link).__name__} link function "


Model key   : sev_fire_incurred
Model label : Severity: fire_incurred
Family      : Gamma
Response    : severity per claim (raw)
Train rows  : 50
Test rows   : 18
--------------------------------------------------------------------------------
TRAIN DIAGNOSTICS
  AIC               : 815.859
  Deviance          : 31.634
  Pearson chi2/df   : 0.777
--------------------------------------------------------------------------------
TEST METRICS (Severity, modeled scale)
  MAE_sev           : 5,171.686106
  RMSE_sev          : 12,379.005423
Model key   : sev_glass_incurred
Model label : Severity: glass_incurred
Family      : Gamma
Response    : severity per claim (raw)
Train rows  : 8447
Test rows   : 2072
--------------------------------------------------------------------------------
TRAIN DIAGNOSTICS
  AIC               : 114,337.118
  Deviance          : 3,803.714
  Pearson chi2/df   : 0.349
--------------------------------------------------------------------------------
TEST METRICS (Sev

c:\Users\eswar\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:308: DomainWarning: The InversePower link function does not respect the domain of the Gamma family.
  warnings.warn((f"The {type(family.link).__name__} link function "
c:\Users\eswar\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:308: DomainWarning: The InversePower link function does not respect the domain of the Gamma family.
  warnings.warn((f"The {type(family.link).__name__} link function "
c:\Users\eswar\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:898: RuntimeWarning: divide by zero encountered in scalar divide
  return np.sum(resid / self.family.variance(mu)) / self.df_resid
c:\Users\eswar\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\genmod\families\family.py:810: RuntimeWarning: divide by zero encountered in log
  ll_ob

In [111]:
import pandas as pd

diag_df = pd.DataFrame(glm_diagnostics).T
diag_df.index.name = "model_key"
diag_df

,model_label,family,type,response_col,exposure_col,offset_col,train_AIC,train_deviance,train_df_resid,train_pearson_chi2,...,test_RMSE_count,test_MAE_freq,test_RMSE_freq,incurred_col,claims_col,response_desc,n_train,n_test,test_MAE_sev,test_RMSE_sev
model_key,,,,,,,,,,,,,,,,,,,,,
freq_total_claims,Frequency: total_claims,NegativeBinomial,frequency,total_claims,total_exposure,log_total_exposure,272326.322,156350.417,276109,395273.344,...,0.549,0.460,1.366,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq_liability_claims,Frequency: liability_claims,Poisson,frequency,liability_claims,liability_exposure,log_liability_exposure,170024.593,124356.208,276109,346211.858,...,0.329,0.245,1.059,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq_property_claims,Frequency: property_claims,NegativeBinomial,frequency,property_claims,total_exposure,log_total_exposure,102725.909,74563.173,276109,587050.789,...,0.344,0.135,0.672,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq_theft_claims,Frequency: theft_claims,Poisson,frequency,theft_claims,total_exposure,log_total_exposure,16771.696,14037.739,276109,278336.075,...,0.072,0.014,0.141,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq_fire_claims,Frequency: fire_claims,Poisson,frequency,fire_claims,total_exposure,log_total_exposure,1001.054,859.054,276109,660151.519,...,0.016,0.001,0.045,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq_glass_claims,Frequency: glass_claims,Poisson,frequency,glass_claims,total_exposure,log_total_exposure,75203.684,58073.352,276109,298452.499,...,0.179,0.086,0.360,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq_legal_protection_claims,Frequency: legal_protection_claims,Poisson,frequency,legal_protection_claims,total_exposure,log_total_exposure,26569.914,21889.344,276109,328117.622,...,0.088,0.024,0.205,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq_occupants_claims,Frequency: occupants_claims,Poisson,frequency,occupants_claims,total_exposure,log_total_exposure,283.970,217.970,276109,179844.037,...,0.004,0.000,0.007,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sev_total_incurred,Severity: total_incurred,Gamma,severity,NaN,NaN,NaN,454748.371,22024.626,30343,17350.322,...,NaN,NaN,NaN,total_incurred,total_claims,severity per claim (raw),30364,7544,392.165,516.268


USER INTERFACE

In [84]:
unique_brands_sorted = sorted(df["vehicle_brand_grp"].unique().tolist())
print(unique_brands_sorted)


['AUDI', 'CITROEN', 'FORD', 'MERCEDES', 'OPEL', 'OTHER', 'PEUGEOT', 'RENAULT', 'SEAT', 'TOYOTA', 'VOLKSWAGEN']
